# Recommender Systems — Week 3 Study Notebook
### Content-based Recommendation

This notebook reworks the professor's `content_based.ipynb`. It uses the same five sections as the Week 2 notebook:

| Section | What you get | Why it is there |
|---|---|---|
| **A. The real-world problem** | A concrete situation from a real product | So you know *why* the method exists |
| **B. Solving it by hand** | Every idea worked out step by step on tiny tables, then the general algorithm | So you can reproduce it on paper in the exam |
| **C. The professor's code** | What his code does, line by line, what is wrong with it, and why it is not written the industrial way | So you can read and answer questions about his code |
| **D. The industry version** | The same job done the way it is done at work, on real data | So you learn the practical way |
| **E. Exam questions** | The professor's 8 exercises with answers, plus likely variations | Exam preparation |

Every number computed by hand is checked by a code cell right after it.

---
# 0. Where we are

## 0.1 The story so far

In Week 2 you were the ML engineer at the streaming company **StreamBox**, and you built:

1. **Baselines** (popularity, averages) for users we know nothing about,
2. **Collaborative filtering** (item-kNN, MF, LightGCN), which learns from *who watched what*.

Collaborative filtering (CF) worked well. It answered our Week 2 question correctly: it predicted that Ana would dislike *Terminator*.

## 0.2 The new problem

On Friday, StreamBox releases a new movie, **Cars**. It is an animated family movie. On Friday morning it has **zero ratings and zero views**.

The product manager asks: *"Put Cars on the home page of the people who will like it."*

Try to do it with Week 2 methods:

- **Item-kNN CF** needs users who watched both Cars and something else. There are none. Every similarity with Cars is 0.
- **MF / LightGCN** learn a vector for Cars only from its interactions. There are none, so its vector is still random.
- **Popularity** gives Cars a count of 0, so it goes to the bottom.

**Every CF method is blind to a new item.** This is called the **item cold-start problem**, and it is the main reason content-based recommendation exists.

Yet a human sees the answer immediately: *Cars is an animated family movie; Ana loves animated family movies; show it to Ana.* The human is not using other users' behaviour. The human is using **what the movie is**: its **content**. That is this week's topic.

## 0.3 The tiny tables used for hand calculations

**Ratings** (the same table as Week 2):

| | Toy Story | Nemo | Matrix | Terminator | Titanic |
|---|---|---|---|---|---|
| **Ana** | 5 | 4 | 1 | ? | ? |
| **Ben** | 1 | ? | 5 | 4 | ? |
| **Cem** | 4 | 5 | ? | 2 | 3 |
| **Dee** | ? | 2 | 4 | 5 | 1 |

**Content: the genres of each movie**, including the new one:

| Movie | Genres |
|---|---|
| Toy Story | Animation, Children, Comedy |
| Nemo | Animation, Children, Adventure |
| Matrix | Action, Sci-Fi |
| Terminator | Action, Sci-Fi, Thriller |
| Titanic | Drama, Romance |
| **Cars** (new, no ratings) | Animation, Children, Comedy |

### Setup

In [1]:
# Colab: run this line once. sentence-transformers is needed for the text-embedding part.
# %pip install -q sentence-transformers implicit faiss-cpu

In [2]:
import os, math, time, heapq, itertools, operator, zipfile, urllib.request, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
SEED = 42
np.random.seed(SEED)

In [3]:
nan = np.nan
T = pd.DataFrame([[5, 4, 1, nan, nan],
                  [1, nan, 5, 4, nan],
                  [4, 5, nan, 2, 3],
                  [nan, 2, 4, 5, 1]],
                 index=["Ana", "Ben", "Cem", "Dee"],
                 columns=["ToyStory", "Nemo", "Matrix", "Terminator", "Titanic"])
GEN = {"ToyStory":   {"Animation", "Children", "Comedy"},
       "Nemo":       {"Animation", "Children", "Adventure"},
       "Matrix":     {"Action", "Sci-Fi"},
       "Terminator": {"Action", "Sci-Fi", "Thriller"},
       "Titanic":    {"Drama", "Romance"},
       "Cars":       {"Animation", "Children", "Comedy"}}      # the new movie: no ratings at all
T

,ToyStory,Nemo,Matrix,Terminator,Titanic
Ana,5.0,4.0,1.0,NaN,NaN
Ben,1.0,NaN,5.0,4.0,NaN
Cem,4.0,5.0,NaN,2.0,3.0
Dee,NaN,2.0,4.0,5.0,1.0


---
# Part 1 — Content-based Recommendation

## 1.A The real-world problem

**Definition.** A content-based recommender recommends items whose **content** (attributes, text, images) is similar to the content of the items the user liked before. It uses:

- **item content:** genres, tags, title, plot summary, director, cast, release year, price, brand, the text of an article, the audio of a song, the pixels of a product photo;
- **the user's own history:** which items *this* user liked.

It does **not** need other users. That is the key difference from collaborative filtering.

**Where content-based recommendation is essential in real products:**

| Situation | Why CF cannot do it | What is used |
|---|---|---|
| **New item** (a new movie, a new product, a new song) | No interactions yet → CF cannot see it | Its content is known from the first second |
| **News and short-lived items** | An article is relevant for a few hours; by the time it has enough clicks, it is old news | Recommendation by the article's text and topic |
| **Job portals, real estate, dating** | Each item (a job ad, a flat) is unique and is taken quickly, so there is never enough interaction data per item | Matching on attributes: skills, location, price |
| **The long tail** | Niche items have very few interactions, so CF rarely recommends them | Content can surface a niche item that matches the user's taste |
| **Explanations** | "500 users like you liked this" is vague | "Because it is an animated comedy, like Toy Story" |
| **Search-like requests** ("a dark comedy heist movie") | CF has no notion of words or themes | Text embeddings |

**What content-based recommendation cannot do well (remember these for the exam):**

1. **No notion of quality.** A terrible animated movie and a masterpiece with the same genres look identical. CF knows which one people actually enjoyed.
2. **Over-specialisation (the filter bubble).** It only recommends "more of the same". A user who watched three animated movies gets only animated movies. It rarely produces a pleasant surprise (low **serendipity**).
3. **It needs good features.** If the content description is poor (only genres), the recommendations are poor. Getting good item features is often most of the work.
4. **User cold start is not solved.** A brand-new *user* has no history, so there is nothing to compare to. (CF has the same problem.)

**How industry uses it:** almost never alone. Real systems are **hybrids**: CF where there is enough interaction data, content where there is not (new items, niche items), and content features fed into the ranking model. Section 1.B, Step 8, explains the types of hybrids.

## 1.B Solving it by hand

### Step 1 — Representing an item

To compare two items by content, each item must first be turned into something we can compute with. There are two families, and each comes with its own similarity measure:

| Representation | Example for Toy Story | Similarity used |
|---|---|---|
| **A set of labels** | {Animation, Children, Comedy} | Jaccard |
| **A vector of numbers** (one-hot genres, TF-IDF of text, embeddings) | [0.9, 0.2, …] | Cosine |

The professor uses one of each: **genre sets with Jaccard**, and **text embeddings with cosine**. Industry also uses **TF-IDF** heavily, so it is covered here too.

### Step 2 — Genre similarity with Jaccard, by hand

The formula is the same as in Week 2, but the sets now contain **genres** instead of users:

$$J(i,j) = \frac{|\text{genres}_i \cap \text{genres}_j|}{|\text{genres}_i \cup \text{genres}_j|}$$

**Important difference from Week 2.** In Week 2 the sets were "users who rated the item", so the similarity came from behaviour and changed every day. Here the sets come from the item description, so the similarity is known **before anyone watches the movie** and never changes.

**Cars vs Toy Story:** common = {Animation, Children, Comedy} = 3; all = {Animation, Children, Comedy} = 3 → $J = 3/3 = 1.00$
**Cars vs Nemo:** common = {Animation, Children} = 2; all = {Animation, Children, Comedy, Adventure} = 4 → $J = 2/4 = 0.50$
**Cars vs Matrix:** common = { } = 0 → $J = 0.00$

### Step 3 — Predicting a rating: the same weighted average as Week 2

$$\hat r_{ui} = \frac{\sum_{j \in I_u} \text{sim}(i,j)\cdot r_{uj}}{\sum_{j \in I_u} \text{sim}(i,j)}$$

Only the similarity changed. The idea: *if the new movie is very similar in content to a movie the user rated, that rating tells us their opinion of the new movie.*

**Ana → Cars** (the cold-start item). Ana rated Toy Story 5, Nemo 4, Matrix 1.

| Movie $j$ Ana rated | Her rating | sim(Cars, $j$) | sim × rating |
|---|---|---|---|
| Toy Story | 5 | 1.00 | 5.00 |
| Nemo | 4 | 0.50 | 2.00 |
| Matrix | 1 | 0.00 | 0.00 |
| **Sum** | | **1.50** | **7.00** |

$$\hat r_{\text{Ana, Cars}} = \frac{7.00}{1.50} = 4.67$$

**Ben → Cars.** Ben rated Toy Story 1, Matrix 5, Terminator 4. sim(Cars, Toy Story) = 1, sim(Cars, Matrix) = 0, sim(Cars, Terminator) = 0.

$$\hat r_{\text{Ben, Cars}} = \frac{1 \times 1 + 0 \times 5 + 0 \times 4}{1 + 0 + 0} = 1.00$$

So: show Cars to Ana, not to Ben. This is the right answer, on a movie with **zero ratings**. No CF method from Week 2 could do this.

**Our Week 2 question, Ana → Terminator**, with content:

- sim(Terminator, Toy Story) = 0/6 = 0; sim(Terminator, Nemo) = 0/6 = 0
- sim(Terminator, Matrix) = common {Action, Sci-Fi} = 2; all {Action, Sci-Fi, Thriller} = 3 → 0.67

$$\hat r = \frac{0\times5 + 0\times4 + 0.67\times1}{0 + 0 + 0.67} = 1.00$$

Correct again, and again without looking at any other user.

**But look at the weakness.** The prediction for Terminator is built from **one** movie only (Matrix). And if a new animated movie were terrible, the method would still predict 4.67 for Ana, because it cannot know that the movie is bad. Content says *what* a movie is, not *how good* it is.

In [4]:
def jaccard_set(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

def content_predict(user, target):
    rated = T.loc[user].dropna()
    sims  = pd.Series({j: jaccard_set(GEN[target], GEN[j]) for j in rated.index})
    print(f"{user} -> {target}:  sims = {sims.round(2).to_dict()}")
    return (sims * rated).sum() / sims.sum() if sims.sum() > 0 else np.nan

for u, i in [("Ana", "Cars"), ("Ben", "Cars"), ("Ana", "Terminator")]:
    print(f"   prediction = {content_predict(u, i):.2f}\n")

Ana -> Cars:  sims = {'ToyStory': 1.0, 'Nemo': 0.5, 'Matrix': 0.0}
   prediction = 4.67

Ben -> Cars:  sims = {'ToyStory': 1.0, 'Matrix': 0.0, 'Terminator': 0.0}
   prediction = 1.00

Ana -> Terminator:  sims = {'ToyStory': 0.0, 'Nemo': 0.0, 'Matrix': 0.67}
   prediction = 1.00



### Step 4 — Vectors and cosine: one-hot genres, TF-IDF, embeddings

Jaccard on genre sets has two problems:

1. **It is coarse.** MovieLens has only 20 genres, so thousands of movies share exactly the same genre set and get similarity 1.00 with each other. The professor's own `knn_map[3]` shows this: all 30 neighbours have similarity 1.0, so the neighbour list is an arbitrary pick among ties.
2. **It only matches exact words.** "Animation" and "Cartoon" would count as completely different.

The fix is to describe each item with a **vector** of numbers and compare vectors with **cosine**. Three common ways to build the vector:

**(a) One-hot genres.** One position per genre, 1 if the movie has it. Toy Story = [Animation 1, Children 1, Comedy 1, Action 0, …]. This is the same information as the set, written as a vector. (`MultiLabelBinarizer` in sklearn; the professor uses it inside `build_knn_map`.)

**(b) TF-IDF of text.** Use words from tags, plot, title. Each word gets a weight:

$$\text{tf-idf}(w, d) = \underbrace{\text{tf}(w,d)}_{\text{how often } w \text{ is in item } d} \times \underbrace{\log\frac{N}{\text{df}(w)}}_{\text{how rare } w \text{ is among all } N \text{ items}}$$

**Why the IDF part matters, by hand.** Three movies with these tags:

| Movie | Tags |
|---|---|
| Toy Story | movie, pixar, toys |
| Cars | movie, pixar, racing |
| Terminator | movie, robots, future |

The word "movie" appears in all 3 items, so it says nothing about any of them.

| Word | In how many items (df) | IDF $= \ln(3/\text{df})$ |
|---|---|---|
| movie | 3 | $\ln(1) = 0$ |
| pixar | 2 | $\ln(1.5) = 0.41$ |
| toys, racing, robots, future | 1 | $\ln(3) = 1.10$ |

- **Without IDF** (plain word counts), Toy Story vs Terminator share "movie" → cosine $= 1/3 = 0.33$. They look related only because of a meaningless word.
- **With IDF**, "movie" gets weight 0 → Toy Story vs Terminator $= 0$. Toy Story vs Cars still share "pixar" → cosine $= \dfrac{0.41^2}{0.41^2 + 1.10^2} = \dfrac{0.16}{1.37} = 0.12$.

So TF-IDF automatically ignores words that everybody uses and emphasises distinctive words. It is the standard, cheap, explainable text representation, and search engines were built on it.

**(c) Text embeddings (the professor's second method).** A pre-trained neural network (a sentence transformer such as `all-MiniLM-L6-v2`) reads the text of the item and outputs a dense vector (384 numbers for this model). It is trained so that **texts with similar meaning get vectors pointing in a similar direction**, even if they use different words ("cartoon" ≈ "animation"; "Pixar" ≈ "Disney"). Individual numbers have no human meaning; only the direction of the whole vector does.

**Cosine by hand with toy 3-number vectors.** Imagine the embedding had only 3 numbers: [family, action, romance].

| Movie | Vector |
|---|---|
| Toy Story | [3, 1, 0] |
| Nemo | [3, 0, 1] |
| Matrix | [1, 3, 0] |
| Terminator | [0, 3, 1] |
| Titanic | [0, 0, 3] |
| Cars | [3, 2, 0] |

$$\cos(\mathbf a, \mathbf b) = \frac{\mathbf a \cdot \mathbf b}{\lVert\mathbf a\rVert\,\lVert\mathbf b\rVert}$$

**Toy Story vs Cars:** dot $= 3\times3 + 1\times2 + 0\times0 = 11$; lengths $\sqrt{9+1+0} = 3.16$ and $\sqrt{9+4+0} = 3.61$ → $\cos = 11 / 11.40 = 0.96$
**Toy Story vs Nemo:** dot $= 9 + 0 + 0 = 9$; lengths $3.16$ and $3.16$ → $\cos = 9/10 = 0.90$
**Toy Story vs Matrix:** dot $= 3 + 3 + 0 = 6$ → $\cos = 6/10 = 0.60$
**Toy Story vs Terminator:** dot $= 0 + 3 + 0 = 3$ → $\cos = 3/10 = 0.30$
**Toy Story vs Titanic:** dot $= 0$ → $\cos = 0.00$

Unlike genre Jaccard, the values are graded: Cars and Nemo are both "very close", but not tied, and Matrix is "somewhat related" rather than simply 0.

**Which one to use?**

| | One-hot genres | TF-IDF | Embeddings |
|---|---|---|---|
| Understands synonyms | no | no | **yes** |
| Explainable | **yes** | **yes** (shared words) | no |
| Cost | trivial | cheap | needs a neural model (GPU helps) |
| Typical use | filters, simple baselines | search, tags, text-heavy items | modern semantic recommendation and search |

In [5]:
V = pd.DataFrame({"ToyStory": [3, 1, 0], "Nemo": [3, 0, 1], "Matrix": [1, 3, 0],
                  "Terminator": [0, 3, 1], "Titanic": [0, 0, 3], "Cars": [3, 2, 0]},
                 index=["family", "action", "romance"]).T.astype(float)
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print("cosine with Toy Story:", {m: round(cos(V.loc["ToyStory"], V.loc[m]), 2) for m in V.index if m != "ToyStory"})

# TF-IDF example (natural log, no smoothing, as in the hand calculation)
docs = {"ToyStory": ["movie", "pixar", "toys"], "Cars": ["movie", "pixar", "racing"], "Terminator": ["movie", "robots", "future"]}
vocab = sorted({w for d in docs.values() for w in d})
df_w  = {w: sum(w in d for d in docs.values()) for w in vocab}
idf   = {w: math.log(len(docs) / df_w[w]) for w in vocab}
tf    = pd.DataFrame([[1.0 if w in d else 0.0 for w in vocab] for d in docs.values()], index=list(docs), columns=vocab)
tfidf = tf * pd.Series(idf)
print("IDF:", {w: round(v, 2) for w, v in idf.items()})
for a, b in [("ToyStory", "Terminator"), ("ToyStory", "Cars")]:
    print(f"{a} vs {b}:  counts cosine = {cos(tf.loc[a], tf.loc[b]):.2f}   tf-idf cosine = {cos(tfidf.loc[a], tfidf.loc[b]):.2f}")

cosine with Toy Story: {'Nemo': 0.9, 'Matrix': 0.6, 'Terminator': 0.3, 'Titanic': 0.0, 'Cars': 0.96}
IDF: {'future': 1.1, 'movie': 0.0, 'pixar': 0.41, 'racing': 1.1, 'robots': 1.1, 'toys': 1.1}
ToyStory vs Terminator:  counts cosine = 0.33   tf-idf cosine = 0.00
ToyStory vs Cars:  counts cosine = 0.67   tf-idf cosine = 0.12


### Step 5 — Top-N, the industrial way: the user profile

The professor produces a Top-N list in two ways, both from Week 2: **Predict & Sort** (predict a rating for every movie, sort) and the **kNN-map** (precompute neighbours, add up similarities). Both work here too.

Content-based recommendation has a third way that is simpler and is the one most used in practice: **build one vector for the user, the "user profile", and compare it with every item.**

1. **Profile** = the average of the vectors of the items the user **liked** (here: rated ≥ 4). Some systems weight by rating or by recency.
2. **Score** of a candidate item = cosine(profile, item vector).
3. Recommend the highest scores among the items the user has not seen.

**By hand for Ana** (toy vectors from Step 4). Ana liked Toy Story (5) and Nemo (4). She disliked Matrix, so it is not in the profile.

$$\text{profile}_{\text{Ana}} = \frac{[3,1,0] + [3,0,1]}{2} = [3,\ 0.5,\ 0.5] \qquad \lVert\text{profile}\rVert = \sqrt{9 + 0.25 + 0.25} = 3.08$$

| Candidate | Vector | Dot with profile | Length | Cosine |
|---|---|---|---|---|
| Cars | [3, 2, 0] | $9 + 1 + 0 = 10$ | 3.61 | $10 / (3.08 \times 3.61) = 0.90$ |
| Terminator | [0, 3, 1] | $0 + 1.5 + 0.5 = 2$ | 3.16 | $2 / (3.08 \times 3.16) = 0.21$ |
| Titanic | [0, 0, 3] | $0 + 0 + 1.5 = 1.5$ | 3.00 | $1.5 / (3.08 \times 3.00) = 0.16$ |

Ana's list: **Cars, Terminator, Titanic.**

**Why industry prefers the profile approach:**

- One vector per user. Scoring all items is one matrix–vector product, $O(n\,d)$, and with an approximate nearest-neighbour index (FAISS) only a tiny part of the catalog is even touched.
- New items need nothing special: as soon as an item has a vector, it can be scored.
- It is exactly the "two-tower" pattern used by large systems: one vector for the user, one for the item, score = dot product. The only difference in big systems is that the towers are learned neural networks.

In [6]:
liked   = T.loc["Ana"][T.loc["Ana"] >= 4].index
profile = V.loc[liked].mean()
print("Ana's profile:", profile.round(2).tolist())
cands = ["Cars", "Terminator", "Titanic"]
print({m: round(cos(profile, V.loc[m]), 2) for m in cands})

Ana's profile: [3.0, 0.5, 0.5]
{'Cars': 0.9, 'Terminator': 0.21, 'Titanic': 0.16}


### Step 6 — The kNN-map with a heap, by hand

The professor builds the kNN-map ($k$ most similar items per item) and mentions a **heap** for keeping the best $k$. Here is how a heap does it.

**The problem.** We scan the similarities of Toy Story to all other movies, one by one, and want to keep the best $k = 2$ without sorting the whole list.

**The trick: a min-heap of size $k$.** A min-heap always has its **smallest** element on top, reachable in $O(1)$. That smallest element is the "current worst of the best". For each new candidate:

- if the heap has fewer than $k$ items → push it;
- else, if the candidate is better than the top (the current worst) → remove the top and push the candidate;
- else → ignore it.

**By hand**, scanning in this order (similarities from Step 4):

| Step | Candidate | Similarity | Heap before | Action | Heap after (top = smallest) |
|---|---|---|---|---|---|
| 1 | Nemo | 0.90 | [ ] | fewer than 2 → push | [0.90] |
| 2 | Matrix | 0.60 | [0.90] | fewer than 2 → push | [**0.60**, 0.90] |
| 3 | Terminator | 0.30 | [0.60, 0.90] | 0.30 < top 0.60 → ignore | [**0.60**, 0.90] |
| 4 | Titanic | 0.00 | [0.60, 0.90] | 0.00 < 0.60 → ignore | [**0.60**, 0.90] |
| 5 | Cars | 0.96 | [0.60, 0.90] | 0.96 > 0.60 → pop 0.60, push 0.96 | [**0.90**, 0.96] |

Result: `knn_map[Toy Story] = [Cars 0.96, Nemo 0.90]`.

**Why a heap.** Each push or pop costs $O(\log k)$. Scanning $c$ candidates costs $O(c \log k)$ time and $O(k)$ memory. Sorting the whole list would cost $O(c \log c)$ time and $O(c)$ memory. With $c$ = 1,000,000 candidates and $k$ = 30, that is the difference between keeping 30 numbers and keeping a million.

(Python's `heapq` is a min-heap, which is why the professor's example `heapq.nsmallest(2, [(10,"a"),(8,"b"),(5,"c")])` returns `[(5,'c'), (8,'b')]`.)

In [7]:
def top_k_heap(candidates, k):
    heap = []
    for name, s in candidates:
        if len(heap) < k:      heapq.heappush(heap, (s, name)); action = "push"
        elif s > heap[0][0]:   heapq.heappushpop(heap, (s, name)); action = "replace top"
        else:                  action = "ignore"
        print(f"{name:10s} {s:.2f}  {action:12s} heap = {sorted(round(x, 2) for x, _ in heap)}")
    return sorted(heap, reverse=True)

order = ["Nemo", "Matrix", "Terminator", "Titanic", "Cars"]
top_k_heap([(m, cos(V.loc["ToyStory"], V.loc[m])) for m in order], k=2)

Nemo       0.90  push         heap = [0.9]
Matrix     0.60  push         heap = [0.6, 0.9]
Terminator 0.30  ignore       heap = [0.6, 0.9]
Titanic    0.00  ignore       heap = [0.6, 0.9]
Cars       0.96  replace top  heap = [0.9, 0.96]


[(0.9647638212377322, 'Cars'), (0.8999999999999998, 'Nemo')]

### Step 7 — Evaluating fairly: time-based split, Hit Ratio and NDCG

**Random split vs time-based split.** Ana's history, in time order:

| Month | Movie | Rating |
|---|---|---|
| January | Toy Story | 5 |
| February | Nemo | 4 |
| March | Matrix | 1 |
| April | Up | 5 |

- A **random split** might put April in the training data and January in the test data. Then the model learns from the future to predict the past. In production that never happens: we always predict what comes *next*. A random split therefore gives optimistic numbers.
- A **time-based leave-one-out split** holds out each user's **last** positive interaction (here: Up, in April) and trains on everything before it. The professor's final evaluation does exactly this (`sort_values('timestamp')` then `groupby('userId').tail(1)`). **This is the industrial way and a good point to mention in the exam.**

**Hit Ratio and NDCG with one test item per user** (the professor's Exercise 8). If the test item is at rank $p$ (1 = top) inside the top-$N$:

$$\text{hit} = 1, \qquad \text{NDCG} = \frac{1}{\log_2(p + 1)}$$

| Rank $p$ | $\log_2(p+1)$ | NDCG |
|---|---|---|
| 1 | $\log_2 2 = 1$ | 1.000 |
| 3 | $\log_2 4 = 2$ | 0.500 |
| 7 | $\log_2 8 = 3$ | 0.333 |
| not in top-$N$ | — | 0 |

**Why the formula is that simple here:** with exactly one relevant item, the ideal list puts it at rank 1, so IDCG $= 1/\log_2 2 = 1$ and NDCG = DCG. With several relevant items (as in Week 2, Part 3) you must divide by the real IDCG.

**Hit Ratio vs NDCG:** a hit at rank 1 and a hit at rank 20 count the same in Hit Ratio. NDCG gives the first 1.0 and the second 0.23. Two methods with the same Hit Ratio can have very different NDCG; the one with higher NDCG puts the right movie higher, where users actually look (Exercise 7).

### Step 8 — Content vs collaborative, and hybrids

| | Collaborative filtering | Content-based |
|---|---|---|
| Learns from | other users' behaviour | the items' own descriptions |
| New item | **cannot** recommend it | **can** (needs only its content) |
| New user | cannot | cannot (needs some history) |
| Knows about quality | **yes** (people liked it) | no |
| Surprising recommendations | **yes** (finds hidden links) | rarely (more of the same) |
| Explainable | partly ("people who liked X") | **yes** ("same genres as X") |
| Needs | a lot of interaction data | good item features |

Since each one is strong exactly where the other is weak, real systems combine them. The main types of **hybrid** recommender:

1. **Weighted:** $\text{score} = \alpha \cdot \text{CF} + (1-\alpha) \cdot \text{content}$.
2. **Switching:** use CF when the item has enough interactions, content otherwise (new items).
3. **Feature combination:** one model that takes both interaction data and item features as input (for example LightFM, or two-tower neural networks with item text as input).
4. **Cascade / two-stage:** one method generates a few hundred candidates, another re-ranks them. The professor describes this one: CF finds candidates, a content model re-ranks.

**By hand — weighted + switching**, $\alpha = 0.6$, for Ana. CF scores exist only for items with interactions:

| Candidate | CF score | Content score | Final score |
|---|---|---|---|
| Up (old, many views) | 0.80 | 0.90 | $0.6 \times 0.80 + 0.4 \times 0.90 = 0.84$ |
| Terminator (old) | 0.10 | 0.21 | $0.6 \times 0.10 + 0.4 \times 0.21 = 0.14$ |
| Cars (new) | — (no data) | 0.90 | switch to content only: **0.90** |

In practice the two scores are first put on the same scale (for example, divided by each user's maximum), otherwise one of them dominates just because its numbers are bigger.

### The algorithms in general form, with their cost

```
CONTENT_PREDICT(u, i):                               # rating prediction
    num <- 0 ; den <- 0
    for each item j rated by u:
        s   <- sim(content[i], content[j])            # Jaccard on sets or cosine on vectors
        num <- num + s * r[u][j]
        den <- den + s
    return num / den   (fallback if den = 0)

CONTENT_TOPN_PROFILE(u, N):                          # Top-N, profile method
    profile <- average of vec[j] over the items j that u liked
    for each item i not seen by u:
        score[i] <- cosine(profile, vec[i])
    return the N items with the highest score

BUILD_KNN_MAP(k):                                    # offline, for "similar items"
    for each item i:
        heap <- empty min-heap
        for each other item j:
            s <- sim(i, j)
            if size(heap) < k: push (s, j)
            else if s > top(heap): replace top with (s, j)
        knn_map[i] <- heap
```

$n$ = items, $d$ = vector length (20 genres, a few thousand TF-IDF words, 384 embedding numbers), $|I_u|$ = items of the user.

| Operation | Cost | Note |
|---|---|---|
| one similarity | $O(g)$ sets, $O(d)$ vectors | |
| `CONTENT_PREDICT` | $O(\lvert I_u\rvert \cdot d)$ | |
| Top-N Predict & Sort | $O(n \cdot \lvert I_u\rvert \cdot d)$ per user | too slow online |
| `BUILD_KNN_MAP` | $O(n^2 d)$ time, $O(nk)$ memory | offline; the item vectors never change, so it is computed once |
| Top-N with the kNN-map | $O(\lvert I_u\rvert \cdot k)$ | |
| Top-N with the profile | $O(\lvert I_u\rvert d + n\,d)$, or about $O(\log n)$ with an ANN index | what is used at scale |

**One advantage over CF that is easy to miss:** content similarities depend only on the items, not on user behaviour. They do not change when users rate, so the kNN-map is built once and only extended when a new item arrives.

## 1.C The professor's code

### Data and the two lookup tables

The professor loads MovieLens (`ratings`, `movies`, `tags`) and builds two dictionaries:

- `rating_map["userId_movieId"] -> rating` — the same table as in Week 2, now built with a fast dictionary comprehension (`zip` over the columns) instead of an `.iloc` loop.
- `movie_genres[movieId] -> set of genres` — e.g. `{1: {'Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'}}`. `g.split('|')` turns the string `"Adventure|Animation|..."` into a list, and `set(...)` makes it a set so that `&` and `|` (intersection, union) can be used for Jaccard.

In [8]:
if not os.path.exists("ml-latest-small/ratings.csv"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml.zip")
    zipfile.ZipFile("ml.zip").extractall()
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies  = pd.read_csv("ml-latest-small/movies.csv")
tags    = pd.read_csv("ml-latest-small/tags.csv")
title   = movies.set_index("movieId").title

rating_map   = {f"{u}_{m}": r for u, m, r in zip(ratings.userId, ratings.movieId, ratings.rating)}
movie_genres = {m: set(g.split("|")) for m, g in zip(movies.movieId, movies.genres)}
print(ratings.shape, movies.shape, tags.shape)
print(list(movie_genres.items())[:2])
print(f"movies with at least one tag: {tags.movieId.nunique()} of {len(movies)}  ({100*tags.movieId.nunique()/len(movies):.0f}%)")

(100836, 4) (9742, 3) (3683, 4)
[(1, {'Fantasy', 'Children', 'Adventure', 'Comedy', 'Animation'}), (2, {'Fantasy', 'Children', 'Adventure'})]
movies with at least one tag: 1572 of 9742  (16%)


Note the last line: **only a small minority of movies have any tag at all.** For most movies the professor's embedding text is just *title + genres*. Keep this in mind when the embedding results come out weaker than you might expect.

### The text embeddings

The professor builds one text per movie, `title + genres + all its tags`, for example
`"Toy Story (1995) Adventure Animation Children Comedy Fantasy pixar pixar fun"`,
and encodes every text with the sentence transformer `all-MiniLM-L6-v2` into a vector of 384 numbers. `movie_embeddings[movieId]` then gives the vector of a movie.

The cell below does the same. **If the transformer cannot be downloaded** (no internet), it falls back to **LSA**: TF-IDF vectors compressed to 128 numbers with a truncated SVD. LSA is the classic pre-neural way to get dense "semantic" text vectors. It is weaker than a transformer, so the embedding numbers in this file are only indicative; run the notebook on Colab to get the real transformer results.

In [9]:
grouped_tags = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(map(str, x)))
movies_enriched = movies.copy()
movies_enriched["tag"] = movies_enriched.movieId.map(grouped_tags).fillna("")
movies_enriched["combined_features"] = (movies_enriched.title + " " +
                                        movies_enriched.genres.str.replace("|", " ", regex=False) + " " +
                                        movies_enriched.tag)
texts = movies_enriched.combined_features.tolist()
print(texts[0])

try:
    from sentence_transformers import SentenceTransformer
    st_model   = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = st_model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
    EMB_NAME   = "Sentence-transformer (all-MiniLM-L6-v2)"
except Exception as e:
    print("Transformer not available here ->", type(e).__name__, "-> using the LSA fallback")
    tfidf_all  = TfidfVectorizer(min_df=2, stop_words="english").fit_transform(texts)
    embeddings = normalize(TruncatedSVD(128, random_state=SEED).fit_transform(tfidf_all))
    EMB_NAME   = "LSA (TF-IDF + SVD) fallback"

movie_embeddings = {m: e for m, e in zip(movies.movieId, embeddings)}
print(EMB_NAME, "| vector length:", embeddings.shape[1])

Toy Story (1995) Adventure Animation Children Comedy Fantasy pixar pixar fun
Transformer not available here -> ModuleNotFoundError -> using the LSA fallback


LSA (TF-IDF + SVD) fallback | vector length: 128


**How to read a 384-number vector** (the professor's explanation, in short): no single number means anything by itself. Meaning is in the **direction** of the whole vector. The model was trained so that texts with similar meaning point in similar directions, which is why cosine similarity is the right comparison.

### The professor's worked example (do this one on paper)

User 1 rated these five movies. What would they rate *From Dusk Till Dawn* (Action, Comedy, Horror, Thriller)? Their real rating was **3**.

| Movie | Genres | User 1's rating | Common with FDTD | All genres | Jaccard |
|---|---|---|---|---|---|
| Toy Story | Adventure, Animation, Children, Comedy, Fantasy | 4 | Comedy → 1 | 5 + 4 − 1 = 8 | 1/8 = 0.125 |
| Grumpier Old Men | Comedy, Romance | 4 | Comedy → 1 | 2 + 4 − 1 = 5 | 1/5 = 0.200 |
| Heat | Action, Crime, Thriller | 4 | Action, Thriller → 2 | 3 + 4 − 2 = 5 | 2/5 = 0.400 |
| Seven | Mystery, Thriller | 5 | Thriller → 1 | 2 + 4 − 1 = 5 | 1/5 = 0.200 |
| Usual Suspects | Crime, Mystery, Thriller | 5 | Thriller → 1 | 3 + 4 − 1 = 6 | 1/6 = 0.167 |

A useful shortcut for the union: $|A \cup B| = |A| + |B| - |A \cap B|$.

- numerator: $0.125\times4 + 0.2\times4 + 0.4\times4 + 0.2\times5 + 0.167\times5 = 0.5 + 0.8 + 1.6 + 1.0 + 0.833 = 4.733$
- denominator: $0.125 + 0.2 + 0.4 + 0.2 + 0.167 = 1.092$
- prediction: $4.733 / 1.092 = 4.34$

The true rating is 3, so the prediction is 1.34 stars too high. **Why?** User 1 gives 4 and 5 to almost everything here, and every similarity is small and similar (0.13 to 0.40), so the weighted average is close to the user's plain average. Genre overlap says "these movies share a Thriller tag", not "this user will enjoy this one". This is the "no notion of quality" weakness in action.

In [10]:
fdtd = movie_genres[70]
rows = []
for m, r in [(1, 4), (3, 4), (6, 4), (47, 5), (50, 5)]:
    g = movie_genres[m]
    rows.append({"movie": title[m], "rating": r, "common": len(g & fdtd), "union": len(g | fdtd), "jaccard": len(g & fdtd) / len(g | fdtd)})
ex = pd.DataFrame(rows)
print(ex.round(3).to_string(index=False))
print("\nprediction =", round((ex.jaccard * ex.rating).sum() / ex.jaccard.sum(), 2), "   true rating =", rating_map["1_70"])

                      movie  rating  common  union  jaccard
           Toy Story (1995)       4       1      8    0.125
    Grumpier Old Men (1995)       4       1      5    0.200
                Heat (1995)       4       2      5    0.400
Seven (a.k.a. Se7en) (1995)       5       1      5    0.200
 Usual Suspects, The (1995)       5       1      6    0.167

prediction = 4.34    true rating = 3.0


### `item_sim` and `content_based_rating_prediction` (the professor's code)

In [11]:
from numpy.linalg import norm

def item_sim(i, j, sim_type="genre"):
    if sim_type == "genre":                                   # Jaccard on genre sets
        gi, gj = movie_genres.get(i, set()), movie_genres.get(j, set())
        union = len(gi | gj)
        return len(gi & gj) / union if union > 0 else 0.0
    elif sim_type == "embedding":                             # cosine on embedding vectors
        ei, ej = movie_embeddings.get(i), movie_embeddings.get(j)
        if ei is None or ej is None: return 0.0
        return np.dot(ei, ej) / (norm(ei) * norm(ej) + 1e-9)  # +1e-9 avoids division by zero
    raise ValueError("sim_type must be 'genre' or 'embedding'")

def content_based_rating_prediction(u, i, sim_type="genre"):
    r, sum_sim = 0, 0
    movies_rated = ratings[ratings["userId"] == u].movieId    # <-- uses the GLOBAL `ratings` (all data)
    for j in movies_rated:
        sim = item_sim(i, j, sim_type=sim_type)
        key = f"{u}_{j}"
        if key in rating_map:                                 # <-- GLOBAL rating_map (all data)
            r += sim * rating_map[key]
            sum_sim += sim
    return r / sum_sim if sum_sim > 0 else 0

print("user 20, movie 101:", round(content_based_rating_prediction(20, 101, "genre"), 3), "(genre)",
      round(float(content_based_rating_prediction(20, 101, "embedding")), 3), "(embedding)")

user 20, movie 101: 3.561 (genre) 3.489 (embedding)


**Reading the code.** `item_sim` is Step 2 (Jaccard on sets) and Step 4 (cosine on vectors). `content_based_rating_prediction` is the weighted average of Step 3, over **all** movies the user rated.

### A serious bug in the professor's evaluation: the answer is inside the input

In the evaluation cell the professor splits `ratings` into `X_train` and `X_test`, but then calls `content_based_rating_prediction(u, i)`, which reads the **global** `ratings` and `rating_map`, i.e. **all** data including the test rating.

Follow what happens for a test pair $(u, i)$:

1. `movies_rated` contains $i$ itself, because the test rating is in the global `ratings`.
2. `item_sim(i, i)` = 1, the largest possible weight.
3. So the true rating $r_{ui}$ enters the weighted average with the highest weight.

The model is shown the answer and then asked to predict it. This is **data leakage**, and it makes the reported MAE (0.68 for genre) look better than it really is. Let us measure how much.

In [12]:
def content_pred_train_only(u, i, sim_type, user_items):
    """Same formula, but only with the user's TRAINING ratings."""
    num = den = 0.0
    for j, r in user_items.get(u, []):
        s = item_sim(i, j, sim_type)
        num += s * r; den += s
    return num / den if den > 0 else np.nan

Xtr, Xte = train_test_split(ratings, test_size=500, random_state=SEED)
user_items_tr = {u: list(zip(g.movieId, g.rating)) for u, g in Xtr.groupby("userId")}
user_mean_tr  = Xtr.groupby("userId").rating.mean()

res_leak = defaultdict(list)
for u, i, r in zip(Xte.userId, Xte.movieId, Xte.rating):
    for st in ["genre", "embedding"]:
        res_leak[f"{st}: professor (leaky)"].append(abs(r - content_based_rating_prediction(u, i, st)))
        p = content_pred_train_only(u, i, st, user_items_tr)
        res_leak[f"{st}: train only (correct)"].append(abs(r - (p if not np.isnan(p) else user_mean_tr.get(u, Xtr.rating.mean()))))
    res_leak["user mean (baseline)"].append(abs(r - user_mean_tr.get(u, Xtr.rating.mean())))
pd.Series({k: np.mean(v) for k, v in res_leak.items()}, name="MAE").round(4).to_frame()

,MAE
genre: professor (leaky),0.6981
genre: train only (correct),0.7231
embedding: professor (leaky),0.6892
embedding: train only (correct),0.7278
user mean (baseline),0.7598


**Reading the result.** Removing the leak makes the error clearly worse for both similarity types: the professor's numbers were optimistic. The honest content prediction is still somewhat better than predicting the user's average, but only somewhat. That matches the hand example: the weights are weak and similar, so the weighted average stays close to the user's mean. Section 1.D shows that it does not beat the bias baseline, which uses no content at all. Content similarity is a weak signal for *exact rating* prediction (the professor's own markdown makes the same point under "Why Embeddings Don't Always Improve Rating Prediction").

**Lesson for the exam and for work:** every function used during evaluation must receive the **training data only**. Passing the data in as an argument, instead of reading global variables, makes this kind of bug much harder to write.

### Top-N in the professor's notebook

**`top_N_pred_sort`** predicts a rating for all ~9,500 unseen movies and sorts them. The top of his list is several movies with a predicted 5.0. A 5.0 happens when a candidate shares genres only with movies the user rated 5, often a single one, so the prediction is just that one rating. These are artifacts of weak evidence, not great recommendations. (Same efficiency problems as Week 2: $O(n \cdot |I_u|)$ similarity computations per user.)

**`build_knn_map`** is better than in Week 2: it computes **all** similarities at once with matrix operations instead of a Python double loop.

- Genre Jaccard: `MultiLabelBinarizer` turns the genre sets into a 0/1 matrix $G$ (movies × 20 genres). Then intersection = $G G^\top$ (the dot product of two 0/1 rows counts common genres), union = $|A| + |B| - $ intersection, Jaccard = intersection / union. This is the vectorised version of Step 2.
- Embeddings: `cosine_similarity(emb_matrix)` gives all cosines at once.
- `np.argpartition(sims, -(K+1))[-(K+1):]` picks the $K+1$ largest values **without sorting the whole row** ($O(n)$ instead of $O(n \log n)$); then only those $K+1$ are sorted. The +1 is because the movie itself is always its own best match and is skipped.

Two problems with it:

1. **Memory.** It builds the full 9,742 × 9,742 matrix, several times over (intersection, union, result), each about 760 MB as 64-bit numbers. That is fine for MovieLens on Colab, but the full matrix grows with $n^2$: for 1 million items it would need about 8 TB. In practice the matrix is computed in chunks, keeping only the top-$k$ of each row, or an approximate index is used (section 1.D).
2. **Ties.** With genre Jaccard, thousands of movies share the same genre set, so `knn_map[3]` is 30 movies all with similarity 1.0, picked arbitrarily among hundreds of ties. The "neighbours" are a random sample of same-genre movies. This alone explains why the genre kNN-map gets a Hit Ratio close to zero.

**`top_N_knn_map` and `add_sims_and_sort`** are the same as in Week 2, with the same two bugs: `itertools.groupby` on an unsorted list does not add up the similarities of the same movie, and movies the user has already seen are not removed, so they waste places in the top-20.

**The final evaluation (time-based leave-one-out) is done the right way:** the test item is each user's **last** positive rating in time, the training data is everything else, and user histories use only ratings ≥ 4. This is the industrial protocol (Step 7). Section 1.D uses exactly this split.

**One inconsistency to be aware of.** The professor's markdown says embeddings reached "Hit Ratio ~0.26 vs ~0.12" for genres. The output of his own cell is 0.040 vs 0.007. The text was probably written for an earlier version of the code. In the exam, rely on the reasoning (embeddings separate movies better than coarse genres), not on those numbers.

### Why this code is fine for a class but not how it is done at work

| What the professor does | Why it is a problem in practice | What is done instead |
|---|---|---|
| Prediction functions read global `ratings` / `rating_map` | Test data leaks into the prediction (measured above) | Pass the training data in explicitly; build all features and statistics from training data only |
| `ratings[ratings.userId == u]` inside every prediction | Scans 100k rows for each prediction | Pre-group the user histories once (he does this in his last cell), or use a sparse matrix |
| `item_sim` recomputes the two vector lengths on every call | Repeated work, millions of times | Normalise all vectors once; then cosine = dot product |
| Top-N by predicting all ~9,500 movies | Far too slow online | User profile + matrix product, or an ANN index |
| Full $n \times n$ similarity matrix in memory | $O(n^2)$ memory | Chunked top-$k$, or FAISS / a vector database |
| Genre Jaccard as the only content | Thousands of ties at 1.0; no quality signal | Richer features (TF-IDF, embeddings) and a popularity / quality signal in the score |
| `groupby` without sorting; seen items not filtered | Wrong scores; wasted slots | Accumulate in a dictionary or sparse product; always mask seen items |

## 1.D The industry version

### Step 1 — Item feature matrices

In practice every item gets a row in a feature matrix, built once, with all rows normalised to length 1. Then **cosine similarity = dot product**, and every computation below is a matrix product.

- `G`: one-hot genres (20 columns)
- `TF`: TF-IDF of the text (title + genres + tags)
- `EMB`: the text embeddings from section 1.C

In [13]:
item_ids = movies.movieId.to_numpy(); n_items = len(item_ids)
user_ids = np.sort(ratings.userId.unique()); n_users = len(user_ids)
i2idx = pd.Series(np.arange(n_items), index=item_ids)
u2idx = pd.Series(np.arange(n_users), index=user_ids)

G   = normalize(csr_matrix(MultiLabelBinarizer().fit_transform(movies.genres.str.split("|")).astype(float)))
TF  = normalize(TfidfVectorizer(min_df=2, stop_words="english").fit_transform(texts))
EMB = normalize(np.asarray(embeddings, dtype=np.float32))
FEATS = {"Genres (one-hot)": G, "TF-IDF (text)": TF, f"Embeddings": EMB}
for k, F in FEATS.items(): print(f"{k:18s} shape={F.shape}")

def similar_items(F, movie_id, n=5):
    v = F[i2idx[movie_id]]
    s = np.asarray((F @ v.T).todense()).ravel() if sp.issparse(F) else F @ v
    s[i2idx[movie_id]] = -1                                   # not itself
    top = np.argsort(-s)[:n]
    return [f"{title[item_ids[j]][:32]} ({s[j]:.2f})" for j in top]

for mid in [1, 2571]:
    print(f"\nMost similar to {title[mid]}:")
    print(pd.DataFrame({k: similar_items(F, mid) for k, F in FEATS.items()}).to_string(index=False))

Genres (one-hot)   shape=(9742, 20)
TF-IDF (text)      shape=(9742, 3719)
Embeddings         shape=(9742, 128)

Most similar to Toy Story (1995):
                       Genres (one-hot)               TF-IDF (text)                              Embeddings
Emperor's New Groove, The (2000) (1.00)   Toy Story 2 (1999) (0.69)               Toy Story 2 (1999) (0.80)
                    Moana (2016) (1.00) Bug's Life, A (1998) (0.52) We're Back! A Dinosaur's Story ( (0.77)
Asterix and the Vikings (Astérix (1.00)   Toy Story 3 (2010) (0.47)                     Balto (1995) (0.73)
Adventures of Rocky and Bullwink (1.00)         Balto (1995) (0.33)               Toy Story 3 (2010) (0.72)
  Tale of Despereaux, The (2008) (1.00)            Up (2009) (0.32) Indian in the Cupboard, The (199 (0.71)

Most similar to Matrix, The (1999):
                       Genres (one-hot)                         TF-IDF (text)                              Embeddings
                Screamers (1995) (1.00)            

**What to notice:** with genres alone, all neighbours have similarity 1.00 (the tie problem). TF-IDF and embeddings separate them, because they also use the title and tags (e.g. "Toy Story 2" shares title words and the "pixar" tag).

### Step 2 — Rating prediction, vectorised and leak-free

The same weighted average, but with precomputed top-$k$ content neighbours ($k = 30$) and only training data. As in Week 2, the prediction is one sparse matrix product: $\hat R = (X\,S^\top) / (B\,S^\top)$.

In [14]:
def topk_sim(F, k=30, chunk=2000):
    """Top-k cosine neighbours of every item (F rows already L2-normalised). Memory O(n*k), never the full n x n matrix."""
    n = F.shape[0]; rows, cols, vals = [], [], []
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        sim = F[s:e] @ F.T
        sim = sim.toarray() if sp.issparse(sim) else np.asarray(sim)
        sim[np.arange(e - s), np.arange(s, e)] = 0            # no self-similarity
        idx = np.argpartition(-sim, k, axis=1)[:, :k]
        rows.append(np.repeat(np.arange(s, e), k)); cols.append(idx.ravel()); vals.append(np.take_along_axis(sim, idx, 1).ravel())
    S = csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(n, n))
    S.data[S.data < 0] = 0; S.eliminate_zeros()
    return S

def to_csr(df, binary=False):
    v = np.ones(len(df)) if binary else df.rating.to_numpy(float)
    return csr_matrix((v, (u2idx[df.userId].to_numpy(), i2idx[df.movieId].to_numpy())), shape=(n_users, n_items))

X_train, X_test = train_test_split(ratings, test_size=0.1, random_state=SEED)
Xtr_m = to_csr(X_train); Btr_m = Xtr_m.copy(); Btr_m.data[:] = 1
tu, ti, y = u2idx[X_test.userId].to_numpy(), i2idx[X_test.movieId].to_numpy(), X_test.rating.to_numpy()
mae  = lambda p: float(np.mean(np.abs(y - p)))
rmse = lambda p: float(np.sqrt(np.mean((y - p) ** 2)))

mu  = X_train.rating.mean()
b_i = X_train.assign(d=X_train.rating - mu).groupby("movieId").d.agg(lambda s: s.sum() / (10 + len(s)))
b_u = X_train.assign(d=X_train.rating - mu - X_train.movieId.map(b_i)).groupby("userId").d.agg(lambda s: s.sum() / (5 + len(s)))
bias = np.clip(mu + X_test.userId.map(b_u).fillna(0).to_numpy() + X_test.movieId.map(b_i).fillna(0).to_numpy(), 0.5, 5)
umean = X_test.userId.map(X_train.groupby("userId").rating.mean()).fillna(mu).to_numpy()

rows = {"Global mean": (np.full(len(y), mu), 1.0), "User mean": (umean, 1.0), "Bias baseline mu+b_u+b_i": (bias, 1.0)}
for name, F in FEATS.items():
    S = topk_sim(F, k=30)
    num = np.asarray((Xtr_m @ S.T)[tu, ti]).ravel(); den = np.asarray((Btr_m @ S.T)[tu, ti]).ravel()
    ok = den > 1e-9
    p = umean.copy(); p[ok] = num[ok] / den[ok]                # fallback: user mean
    rows[f"Content kNN | {name}"] = (p, ok.mean())
pd.DataFrame({k: {"MAE": mae(p), "RMSE": rmse(p), "coverage": c} for k, (p, c) in rows.items()}).T

,MAE,RMSE,coverage
Global mean,0.8351,1.0506,1.0000
User mean,0.7432,0.9563,1.0000
Bias baseline mu+b_u+b_i,0.6796,0.8861,1.0000
Content kNN | Genres (one-hot),0.7831,1.0440,0.6867
Content kNN | TF-IDF (text),0.7528,1.0072,0.7552
Content kNN | Embeddings,0.7692,1.0214,0.7615


**Reading the table.** This version uses only the $k = 30$ nearest content neighbours of the target movie that the user has rated. Often that is just one or two movies, so the prediction rests on one or two ratings and is noisy: it is even worse than the user mean. The professor's version (all rated movies, section 1.C) was slightly better than the user mean. **Neither beats the bias baseline**, which uses no content at all. This is the professor's point, now measured without leakage: **content similarity is weak at predicting exact ratings.** Its value is elsewhere: in ranking with a quality signal, and above all in cold start (Steps 3 to 5).

### Step 3 — Top-N with the professor's time-based split

Each user's last positive rating (rating > 4) is the test item; everything before it is training data. Every method ranks the **full catalog** with already-seen movies removed, and we report HR@20 and NDCG@20.

The content method is the **user profile** of section 1.B (Step 5): the average of the vectors of the movies the user rated ≥ 4.

In [15]:
pos_sorted = ratings.query("rating > 4").sort_values("timestamp")
te = pos_sorted.groupby("userId").tail(1)                    # last positive per user = test
tr = ratings.drop(te.index)
B_pos  = to_csr(tr[tr.rating >= 4], binary=True)             # positive history (the profile is built from this)
B_seen = to_csr(tr, binary=True)                             # everything seen (masked at recommendation time)
te_u, te_i = u2idx[te.userId].to_numpy(), i2idx[te.movieId].to_numpy()

def evaluate_topn(scores, K=20):
    sc = np.array(scores, dtype=float)
    sc[B_seen.nonzero()] = -np.inf
    rank = (sc[te_u] > sc[te_u, te_i][:, None]).sum(1)        # 0 = top of the list
    return {f"HR@{K}": float((rank < K).mean()), f"NDCG@{K}": float(np.where(rank < K, 1 / np.log2(rank + 2), 0).mean())}

def profile_scores(F, B):
    P = normalize(B @ F)                                      # user profile = (normalised) sum of liked item vectors
    S = P @ F.T
    return S.toarray() if sp.issparse(S) else np.asarray(S)

topn = {}
pop = np.asarray(B_seen.sum(0)).ravel()
topn["Popularity"] = evaluate_topn(np.tile(pop, (n_users, 1)))
content_sc = {}
for name, F in FEATS.items():
    content_sc[name] = profile_scores(F, B_pos)
    topn[f"Content profile | {name}"] = evaluate_topn(content_sc[name])
pd.DataFrame(topn).T

,HR@20,NDCG@20
Popularity,0.1355,0.0564
Content profile | Genres (one-hot),0.0385,0.0279
Content profile | TF-IDF (text),0.0151,0.0074
Content profile | Embeddings,0.0318,0.0118


**Reading the table.** On users with a long history, pure content-based ranking is **below popularity**. Why? It has no notion of quality or popularity: among the thousands of movies whose content matches your taste, it cannot tell the famous good ones from the obscure bad ones (the professor's "concept of quality" point).

**The common industrial fix:** add a quality/popularity signal to the content score. Below: content score + a log-popularity term.

In [16]:
pop_log = np.log1p(pop) / np.log1p(pop).max()                # popularity on a 0..1 scale
for name in FEATS:
    topn[f"Content profile + popularity | {name}"] = evaluate_topn(content_sc[name] + 1.0 * pop_log[None, :])
pd.DataFrame(topn).T

,HR@20,NDCG@20
Popularity,0.1355,0.0564
Content profile | Genres (one-hot),0.0385,0.0279
Content profile | TF-IDF (text),0.0151,0.0074
Content profile | Embeddings,0.0318,0.0118
Content profile + popularity | Genres (one-hot),0.0936,0.0383
Content profile + popularity | TF-IDF (text),0.1237,0.0447
Content profile + popularity | Embeddings,0.1304,0.0490


Adding the popularity term multiplies the content HR by roughly 2.5 to 8 and brings them close to the popularity baseline. Content plus a quality signal is a usable recommender; content alone is not.

### Step 4 — Comparison with collaborative filtering, and two hybrids

Now the item-kNN CF of Week 2 (via the `implicit` library) on exactly the same split, and two hybrids:

- **Weighted:** scale both scores to 0..1 per user, then $0.7 \cdot \text{CF} + 0.3 \cdot \text{content}$ over **all** items.
- **Cascade (two-stage):** CF chooses its 500 best candidates; content only **re-ranks** those: $\text{CF} + 0.3 \cdot \text{content}$ inside the candidate set. This is the architecture the professor describes ("CF generates candidates, a content model re-ranks").

In [17]:
from implicit.nearest_neighbours import CosineRecommender

cf_model = CosineRecommender(K=30)
cf_model.fit(B_pos, show_progress=False)
ids, sc = cf_model.recommend(np.arange(n_users), B_pos, N=500, filter_already_liked_items=True)
sc = np.maximum(sc, 0)                                        # implicit pads short lists with -3.4e38; clip so that scaling works
cf_scores = np.zeros((n_users, n_items)); np.put_along_axis(cf_scores, ids, sc, axis=1)
topn["Item-kNN CF (Week 2)"] = evaluate_topn(cf_scores)

def per_user_minmax(S):
    S = np.array(S, dtype=float); mx = S.max(1, keepdims=True); mn = S.min(1, keepdims=True)
    return (S - mn) / np.where(mx > mn, mx - mn, 1)

emb_key = "Embeddings"
cf_n, ct_n = per_user_minmax(cf_scores), per_user_minmax(content_sc[emb_key])
topn["Hybrid, weighted over all items"] = evaluate_topn(0.7 * cf_n + 0.3 * ct_n)
topn["Hybrid, cascade (content re-ranks CF candidates)"] = evaluate_topn(np.where(cf_scores > 0, cf_n + 0.3 * ct_n, -1.0))
pd.DataFrame(topn).T

,HR@20,NDCG@20
Popularity,0.1355,0.0564
Content profile | Genres (one-hot),0.0385,0.0279
Content profile | TF-IDF (text),0.0151,0.0074
Content profile | Embeddings,0.0318,0.0118
Content profile + popularity | Genres (one-hot),0.0936,0.0383
Content profile + popularity | TF-IDF (text),0.1237,0.0447
Content profile + popularity | Embeddings,0.1304,0.0490
Item-kNN CF (Week 2),0.2057,0.0847
"Hybrid, weighted over all items",0.2191,0.0879
"Hybrid, cascade (content re-ranks CF candidates)",0.2174,0.0885


**Reading the table.**

- **CF wins clearly when there is plenty of interaction data.** This is the professor's section "Why does Item-Based CF outperform Content-Based?": real behaviour carries taste and quality signals that titles, genres and a few tags do not.
- **Both hybrids are slightly better than CF alone** (about 0.22 vs 0.21 HR@20). Content adds a little information about the order of CF's candidates. The gain is small and comes from a single split, so treat it as a direction, not a precise number.
- **The cascade is the version used at scale:** content scores are needed only for the few hundred CF candidates, not for the whole catalog, and CF guarantees that every candidate has a quality signal.
- **Scaling the scores is not a detail.** `implicit` fills short candidate lists with the value −3.4·10³⁸. Before this was clipped (the `np.maximum(sc, 0)` line), the min–max scaling collapsed all CF scores to almost the same value and the hybrids looked far worse than CF. When combining scores from different models, always check their ranges first.
- So why do companies bother with content at all if the gain is small for old items? Step 5 answers that.

### Step 5 — Where content wins: new items (cold start)

**The experiment.** Pretend 10% of the movies are brand new: remove **all** their ratings from the training data. Then build a "New releases for you" row: rank only these new movies for each user. The test item is the user's last positive rating on a new movie.

- CF has never seen these movies: all its scores are 0, so its ranking is effectively random.
- Content methods only need the movies' text and genres, which exist from day one.

In [18]:
rng = np.random.default_rng(SEED)
cnt = ratings.groupby("movieId").size()
NEW = rng.choice(cnt[cnt >= 5].index.to_numpy(), size=(cnt >= 5).sum() // 10, replace=False)    # the "new" movies
new_idx = i2idx[NEW].to_numpy(); pos_in_new = pd.Series(np.arange(len(new_idx)), index=new_idx)

tr_cold = ratings[~ratings.movieId.isin(NEW)]                                                  # no ratings at all for NEW movies
te_cold = (ratings[ratings.movieId.isin(NEW) & (ratings.rating >= 4)]
           .sort_values("timestamp").groupby("userId").tail(1))
B_cold  = to_csr(tr_cold[tr_cold.rating >= 4], binary=True)
print(f"new movies: {len(NEW)}   test users: {len(te_cold)}   random HR@10 = 10/{len(NEW)} = {10/len(NEW):.3f}")

def evaluate_new(scores, K=10):
    sc = np.asarray(scores, dtype=float)[:, new_idx] + rng.random((n_users, len(new_idx))) * 1e-9   # random tie-break
    u = u2idx[te_cold.userId].to_numpy(); t = pos_in_new[i2idx[te_cold.movieId].to_numpy()].to_numpy()
    rank = (sc[u] > sc[u, t][:, None]).sum(1)
    return {f"HR@{K}": float((rank < K).mean()), f"NDCG@{K}": float(np.where(rank < K, 1 / np.log2(rank + 2), 0).mean())}

cold = {"Random": evaluate_new(np.zeros((n_users, n_items)))}
m_cold = CosineRecommender(K=30); m_cold.fit(B_cold, show_progress=False)
ids, sc = m_cold.recommend(np.arange(n_users), B_cold, N=1000, filter_already_liked_items=True)
sc = np.maximum(sc, 0)
cf_cold = np.zeros((n_users, n_items)); np.put_along_axis(cf_cold, ids, sc, axis=1)
cold["Item-kNN CF"] = evaluate_new(cf_cold)
for name, F in FEATS.items():
    cold[f"Content profile | {name}"] = evaluate_new(profile_scores(F, B_cold))
pd.DataFrame(cold).T

new movies: 365   test users: 553   random HR@10 = 10/365 = 0.027


,HR@10,NDCG@10
Random,0.0289,0.0122
Item-kNN CF,0.0271,0.0104
Content profile | Genres (one-hot),0.0506,0.0241
Content profile | TF-IDF (text),0.0796,0.0393
Content profile | Embeddings,0.1555,0.0812


**This table is the reason content-based recommendation exists.** On new items CF is no better than random, while content methods are several times better than random, with the richest text representation doing best. In production this is the **switching hybrid**: CF for items with enough history, content for new and niche items.

### Step 6 — Serving at scale: the user profile in an ANN index

With millions of items you do not compute a score for every item. Item vectors go into an **approximate nearest-neighbour (ANN) index**; at request time the user's profile vector is the query and the index returns the closest items in milliseconds. FAISS (Meta) is the standard library; managed "vector databases" (pgvector in PostgreSQL, Elasticsearch / OpenSearch k-NN, Milvus, Pinecone, Qdrant) do the same as a service.

In [19]:
import faiss
index = faiss.IndexHNSWFlat(EMB.shape[1], 32, faiss.METRIC_INNER_PRODUCT)   # graph-based ANN index; inner product = cosine (normalised)
index.add(EMB.astype("float32"))

u = u2idx[1]
profile_u = normalize(np.asarray(B_pos[u] @ EMB)).astype("float32")
scores_u, idx_u = index.search(profile_u, 30)
seen_u = set(B_seen[u].indices)
recs = [j for j in idx_u[0] if j not in seen_u][:8]
print("User 1 liked e.g.:", "; ".join(title[item_ids[j]] for j in B_pos[u].indices[:5]))
print("ANN content recommendations:"); [print("  ", title[item_ids[j]]) for j in recs];

User 1 liked e.g.: Toy Story (1995); Grumpier Old Men (1995); Heat (1995); Seven (a.k.a. Se7en) (1995); Usual Suspects, The (1995)
ANN content recommendations:
   Aelita: The Queen of Mars (Aelita) (1924)
   Super Mario Bros. (1993)
   Who Am I? (Wo shi shei) (1998)
   Laputa: Castle in the Sky (Tenkû no shiro Rapyuta) (1986)
   Mighty Joe Young (1998)
   Maximum Ride (2016)
   Batman/Superman Movie, The (1998)
   Honey, We Shrunk Ourselves (1997)


The point of this cell is the **mechanics** (build an index once, query it with a user vector, filter what was already seen), not the quality of this particular list. As Step 3 showed, pure content retrieval needs a quality or CF signal before the final ranking, which is exactly what the cascade does.

### Industry summary

| Need | Tool |
|---|---|
| Text → vectors (classic) | `TfidfVectorizer` (scikit-learn); Elasticsearch / OpenSearch "more like this" |
| Text → vectors (semantic) | `sentence-transformers` (open models such as all-MiniLM, E5, BGE), or hosted embedding APIs |
| Images, audio | pre-trained vision / audio models (e.g. CLIP for images) produce the item vectors in the same way |
| Nearest-neighbour search | FAISS, ScaNN, HNSW; vector databases (pgvector, Milvus, Qdrant, Pinecone) |
| CF and content in one model | LightFM (MF with item features), two-tower models (TensorFlow Recommenders, TorchRec) |
| Typical architecture | offline: compute item vectors and the index → online: user profile → ANN retrieval → ranker that also uses CF and popularity signals → business rules |

## 1.E Exam — Content-based recommendation

### The professor's exercises

**Exercise 1 — Basic rating prediction.** *Predict User 2's rating for Toy Story (movieId 1) with `sim_type='embedding'`.*

Call the function exactly as written. (His output with the transformer is 3.92. Here the number differs if the LSA fallback was used.) Note that if User 2 had rated Toy Story, the leak from section 1.C would put the true rating inside the prediction.

In [20]:
print("Exercise 1:", round(float(content_based_rating_prediction(2, 1, sim_type="embedding")), 2))

Exercise 1: 4.36


**Exercise 2 — kNN rating prediction.** *Use only the top $k$ most similar movies the user has rated.*

Algorithm: compute the similarity to every movie the user rated, keep the $k$ largest (a heap of size $k$ does this in $O(|I_u| \log k)$; `heapq.nlargest` uses one), and take the weighted average over those $k$ only.

In [21]:
def content_based_rating_prediction_knn(u, i, sim_type="embedding", k=5):
    rated = [(item_sim(i, j, sim_type), rating_map[f"{u}_{j}"]) for j in ratings.loc[ratings.userId == u, "movieId"]]
    top = heapq.nlargest(k, rated, key=lambda t: t[0])         # k most similar rated movies
    den = sum(s for s, _ in top)
    return sum(s * r for s, r in top) / den if den > 0 else np.nan

print("Exercise 2 (k=5):", round(float(content_based_rating_prediction_knn(2, 1, "embedding", k=5)), 2))

Exercise 2 (k=5): 4.62


**Exercise 3 — Find the 5 most similar movies to The Matrix (2571), excluding itself.** With normalised vectors this is one matrix–vector product and a sort, i.e. `similar_items` from section 1.D:

In [22]:
print("Exercise 3:", similar_items(EMB, 2571, n=5))

Exercise 3: ['eXistenZ (1999) (0.94)', 'Wing Commander (1999) (0.91)', 'Thirteenth Floor, The (1999) (0.88)', 'Universal Soldier: The Return (1 (0.88)', 'Ready Player One (0.87)']


**Exercise 4 — Cold start: a brand-new movie with zero ratings. Which approach can recommend it, CF or content-based? Why?**
Content-based. Collaborative filtering computes item similarity (or learns item vectors) from user interactions; a movie with no interactions has no similarity to anything and a random or empty vector, so CF cannot rank it. A content-based method only needs the item's own description (genres, title, tags, plot), which is available at the moment the item is added, so it can compare the new movie with what each user liked immediately. (Section 1.D, Step 5 measures this: CF ≈ random on new items, content several times better.)

**Exercise 5 — Why can embeddings have a much better Hit Ratio than genres while their MAE is similar?**
MAE and Hit Ratio measure different abilities. MAE only checks how close the predicted rating is to the true one on items the user rated; ratings are concentrated around the user's average, and both similarity types act only as weights in a weighted average, so both predictions end up near the user's mean and get similar MAE. Hit Ratio requires ranking the right movie above thousands of others. Genres are coarse: thousands of movies share the same genre set and get tied scores, so the relevant movie is buried among ties. Embeddings produce graded, finer similarities (they also use title words and tags and understand related meanings), so they can separate movies and lift the relevant one into the top-N.

**Exercise 6 — Two extra pieces of metadata that would improve the embeddings.**
The **plot summary / synopsis** (rich semantic description of story, setting and tone) and the **director and main cast** (strong indicators of style and audience). Other good answers: keywords/themes, user reviews, release year/era, language, production studio.

**Exercise 7 — Same Hit Ratio@100, but Method A has much higher NDCG@100. What does it mean?**
Both methods place the relevant movie somewhere in the top 100 equally often, but Method A places it **higher** in the list. Hit Ratio is binary (a hit at rank 1 counts the same as at rank 100); NDCG discounts each hit by $1/\log_2(\text{rank}+1)$, so it rewards hits near the top. Since users mostly look at the first few positions, Method A is more useful in practice.

**Exercise 8 — NDCG for a single hit.**
With one relevant item the ideal DCG is 1, so NDCG $= 1/\log_2(\text{rank}+1)$:

1. rank 3: $1/\log_2 4 = 1/2 = 0.5$
2. rank 1: $1/\log_2 2 = 1$
3. rank 7: $1/\log_2 8 = 1/3 = 0.333$

(Careful: this shortcut is only valid with **one** relevant item. With several, divide DCG by the real IDCG.)

### The professor's in-text question: "Can you suggest a solution?" (efficiency)

> Do not predict a rating for every item. (1) Precompute item–item content similarities offline and keep only the top-$k$ neighbours per item; since content does not change, this is computed once and only extended for new items. (2) At request time, either aggregate the neighbours of the user's items (kNN-map), or build a user-profile vector and query an approximate nearest-neighbour index (FAISS/HNSW) to get a few hundred candidates, then rank only those.

### Likely variations — calculations (do them on paper)

**V1. Jaccard on genres and a prediction.** A user rated A {Comedy, Romance} = 5, B {Action, Thriller} = 2, C {Comedy, Drama} = 4. Predict the rating of X {Comedy, Romance, Drama}.

- $J(X,A) = 2/3 = 0.667$ (common Comedy, Romance; all Comedy, Romance, Drama)
- $J(X,B) = 0/5 = 0$
- $J(X,C) = 2/3 = 0.667$ (common Comedy, Drama; all Comedy, Romance, Drama)
- $\hat r = (0.667\times5 + 0\times2 + 0.667\times4)/(0.667 + 0 + 0.667) = 6.0/1.333 = 4.5$

**V2. TF-IDF.** Three documents: d1 = {cat, dog}, d2 = {cat, fish}, d3 = {cat, bird}. Which word has IDF 0 and why?
"cat" appears in all 3 documents: $\ln(3/3) = 0$. A word that appears everywhere does not help to tell items apart, so TF-IDF gives it no weight. The others get $\ln(3/1) = 1.10$.

**V3. User profile.** A user liked items with vectors [1, 0] and [0.6, 0.8]. Score candidates P = [1, 1] and Q = [0, 1].
Profile $= ([1,0] + [0.6,0.8])/2 = [0.8, 0.4]$, length $\sqrt{0.64 + 0.16} = 0.894$.
$\cos(\text{profile}, P) = (0.8 + 0.4)/(0.894 \times 1.414) = 1.2/1.265 = 0.949$; $\cos(\text{profile}, Q) = 0.4/(0.894 \times 1) = 0.447$ → recommend P first.

**V4. Heap trace.** Keep the top 3 of the similarities 0.2, 0.9, 0.5, 0.1, 0.7, 0.8 with a min-heap.
push 0.2 → [0.2]; push 0.9 → [0.2, 0.9]; push 0.5 → [0.2, 0.5, 0.9]; 0.1 < 0.2 ignore; 0.7 > 0.2 replace → [0.5, 0.7, 0.9]; 0.8 > 0.5 replace → [0.7, 0.8, 0.9]. Result: 0.9, 0.8, 0.7.

**V5. NDCG with two relevant items.** Relevant items at ranks 2 and 4 of a top-5 list; the user has 2 relevant items.
DCG $= 1/\log_2 3 + 1/\log_2 5 = 0.631 + 0.431 = 1.062$. IDCG $= 1 + 0.631 = 1.631$. NDCG $= 0.651$.

**V6. Hybrid.** CF score 0.6 and content score 0.9 for an old item; a new item has only content score 0.8. With a weighted hybrid $\alpha = 0.5$ and switching for new items: old item $= 0.5 \times 0.6 + 0.5 \times 0.9 = 0.75$; new item $= 0.8$ (content only).

### Likely variations — concepts

**C1. Define content-based recommendation and contrast it with collaborative filtering.**
Content-based recommends items whose features are similar to the features of items the user liked, using only the item descriptions and the user's own history. CF recommends using the behaviour of other users (items liked by similar users / items co-liked with the user's items) and needs no item features. Content handles new items and is explainable; CF captures quality and non-obvious relations but needs interaction data.

**C2. List the main limitations of content-based recommenders.**
(1) No notion of quality: similar content ≠ good. (2) Over-specialisation / filter bubble: recommends more of the same, low serendipity. (3) Depends on the quality of features; feature engineering is costly. (4) Does not solve user cold start. (5) Coarse features (e.g. genres) create many ties.

**C3. Why does item-based CF outperform content-based methods when plenty of ratings exist?** (the professor's section)
Behaviour is a stronger signal than metadata: it captures unspoken preferences and quality (people's choices reflect what is actually good), finds non-obvious relations between different genres, and does not suffer from sparse or shallow item descriptions.

**C4. If CF is better with enough data, is content still useful?** (the professor's section)
Yes: long-tail items with few ratings, explainability ("shares the cyberpunk theme with The Matrix"), mood or query-based requests, and as features or a re-ranking stage in hybrid systems.

**C5. Name and explain the types of hybrid recommenders.**
Weighted (linear combination of scores), switching (choose CF or content depending on the data available for the item/user), feature combination (one model using both interactions and item features, e.g. LightFM, two-tower), cascade / two-stage (one method generates candidates, another re-ranks).

**C6. Why is a time-based split better than a random split for evaluation?**
In production we always predict future behaviour from past behaviour. A random split lets the model train on interactions that happened after the test interaction (using the future to predict the past), which inflates offline results. A time-based split (e.g. last interaction per user as test) reproduces the real situation.

**C7. Why does the genre-based kNN map perform so poorly in Top-N?**
MovieLens has only ~20 genres, so thousands of movies share the same genre set and all have Jaccard similarity 1.0 with each other. The top-30 neighbours are an arbitrary pick among hundreds of ties, carrying no information about which of them the user would choose; and genre carries no quality signal.

**C8. What is wrong with the professor's rating-prediction evaluation?** (code reading)
The prediction function reads the global `ratings` and `rating_map`, which contain the test rating. The test movie itself is among the user's rated movies with similarity 1, so its true rating enters the weighted average with the highest weight. This is data leakage; it makes the MAE optimistic. Fix: pass only the training data to the prediction function.

**C9. Why is a heap used to build the kNN-map? Give the complexity.**
To keep the $k$ best of $c$ candidates without sorting all of them: a min-heap of size $k$ keeps the current worst-of-the-best on top; each candidate costs $O(\log k)$ → $O(c \log k)$ time and $O(k)$ memory, versus $O(c \log c)$ time and $O(c)$ memory for a full sort.

**C10. How do you make cosine similarity cheap for many items?**
Normalise every item vector to length 1 once. Then cosine = dot product, and all similarities between a set of items and all items are one matrix product, which runs in optimised compiled code (BLAS/GPU), and can be served with an ANN index.

---
# 2. Summary and exam-night review

### Week 2 and Week 3 together

| | Baselines | Collaborative filtering (kNN, MF, LightGCN) | Content-based |
|---|---|---|---|
| Uses | counts and averages | other users' behaviour | the items' own descriptions |
| Personalised | no | yes | yes |
| New item | no | **no** | **yes** |
| New user | yes (popularity) | no | no |
| Quality signal | yes (popularity, ratings) | yes | **no** |
| Serendipity | none | high | low (filter bubble) |
| Explainable | yes | partly | yes |
| Typical role in industry | fallback, feature | main engine when data is rich | cold start, long tail, re-ranking, explanations |

### Formulas to know by heart

- Jaccard on sets: $J = \dfrac{|A \cap B|}{|A \cup B|}$, with $|A \cup B| = |A| + |B| - |A \cap B|$
- Cosine: $\dfrac{\mathbf a \cdot \mathbf b}{\lVert\mathbf a\rVert \lVert\mathbf b\rVert}$ (= dot product for normalised vectors)
- TF-IDF: $\text{tf}(w,d) \times \log\dfrac{N}{\text{df}(w)}$
- Prediction: $\hat r_{ui} = \dfrac{\sum_j \text{sim}(i,j)\, r_{uj}}{\sum_j \text{sim}(i,j)}$
- User profile: average of the vectors of liked items; score = cosine(profile, item)
- Single-hit NDCG: $1/\log_2(\text{rank} + 1)$ → rank 1: 1, rank 2: 0.631, rank 3: 0.5, rank 7: 0.333

### Common mistakes

1. Forgetting that the union in Jaccard counts shared elements once.
2. Using the single-hit NDCG shortcut when there are several relevant items.
3. Saying content-based solves the **user** cold start. It solves the **item** cold start.
4. Concluding "embeddings are useless" because MAE did not improve: MAE and ranking metrics measure different things.
5. Trusting an evaluation where the test data can reach the model (leakage).

### Practice (to make the algorithms stick)

- Redo the From Dusk Till Dawn prediction (section 1.C) and the Cars predictions (Step 3) without looking.
- Trace the heap of Step 6 with $k = 3$.
- In section 1.D, change the weight of the popularity term (0.3, 1, 3) and the cascade weight, and watch HR@20.
- On Colab, run the notebook with the real sentence transformer and compare the embedding rows with the LSA rows saved here.